# Abalone Age Regression - Multi-Layer Perceptron (MLP)

## Project Overview

This notebook demonstrates how to build and train a Multi-Layer Perceptron (MLP) to predict the age of abalone (shellfish) from physical measurements using PyTorch.

### Learning Objectives:
- Regression loss: Mean Squared Error (MSE)
- Handling numeric features in tabular data
- Feature normalization and scaling techniques
- Understanding overfitting vs regularization
- L1 and L2 regularization
- Complete regression workflow from problem definition to deployment

### Dataset:
- **Abalone Dataset (UCI)**: Classic regression dataset
- **Target**: Age of abalone (rings + 1.5, as rings represent age)
- **Features**: Length, diameter, height, whole weight, shucked weight, viscera weight, shell weight, sex
- **Task**: Regression (predict continuous value)

---


## Part 1: Problem Definition

### Understanding the Task

**Problem Type**: Regression
- **Input**: Physical measurements of abalone (8 features)
- **Output**: Age of abalone (continuous value)
- **Goal**: Predict age accurately from physical characteristics

### Abalone Features:
1. **Sex**: M, F, I (infant) - categorical, needs encoding
2. **Length**: Longest shell measurement
3. **Diameter**: Perpendicular to length
4. **Height**: With meat in shell
5. **Whole weight**: Whole abalone
6. **Shucked weight**: Weight of meat
7. **Viscera weight**: Gut weight (after bleeding)
8. **Shell weight**: After being dried

### Target Variable:
- **Rings**: Number of rings (indicates age)
- **Age**: Rings + 1.5 (actual age in years)

### Key Considerations:
1. **Numeric Features**: All features except sex are numeric
2. **Feature Scaling**: Critical for regression with MLP
3. **Overfitting**: Risk with small dataset (~4,177 samples)
4. **Regularization**: Essential to prevent overfitting
5. **Loss Function**: MSE for regression

### Success Criteria:
- Low Mean Squared Error (MSE)
- Low Mean Absolute Error (MAE)
- Good generalization (validation vs training)
- R² score > 0.5


## Part 2: Data Collection

### Abalone Dataset

The Abalone dataset is a classic regression benchmark:
- **4,177 samples**
- **8 input features** (7 numeric + 1 categorical)
- **1 target variable** (age/rings)
- **Source**: UCI Machine Learning Repository

### Why This Dataset is Good for MLP:
- **Clean numeric features**: Well-structured tabular data
- **Non-linear patterns**: Physical measurements have complex relationships
- **Appropriate size**: Not too small, not too large
- **Real-world application**: Practical use case

We'll download it from UCI or use a local copy.


In [ ]:
# Import necessary libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from typing import Tuple, Dict, List
import time
import urllib.request
import os

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check if CUDA is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Download Abalone dataset from UCI
def download_abalone_data(url: str = "https://archive.ics.uci.edu/ml/machine-learning-databases/abalone/abalone.data", 
                         filename: str = "abalone.data") -> str:
    """
    Download Abalone dataset from UCI repository.
    
    Args:
        url: URL to download the dataset
        filename: Local filename to save
        
    Returns:
        Path to downloaded file
    """
    if not os.path.exists(filename):
        print(f"Downloading Abalone dataset from {url}...")
        urllib.request.urlretrieve(url, filename)
        print(f"Downloaded to {filename}")
    else:
        print(f"Dataset already exists at {filename}")
    return filename

# Column names for Abalone dataset
abalone_columns = [
    'Sex', 'Length', 'Diameter', 'Height', 
    'Whole_weight', 'Shucked_weight', 'Viscera_weight', 
    'Shell_weight', 'Rings'
]

# Download and load data
data_file = download_abalone_data()
df = pd.read_csv(data_file, names=abalone_columns)

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nDataset info:")
print(df.info())


## Part 3: Data Understanding

Let's explore the dataset to understand its characteristics, distributions, and relationships.


In [ ]:
def explore_dataset(df: pd.DataFrame) -> None:
    """
    Explore and visualize dataset characteristics.
    
    Args:
        df: DataFrame containing the dataset
    """
    print(f"{'='*60}")
    print(f"Dataset Exploration")
    print(f"{'='*60}")
    print(f"Shape: {df.shape}")
    print(f"\nMissing values:")
    print(df.isnull().sum())
    print(f"\nData types:")
    print(df.dtypes)
    print(f"\nBasic statistics:")
    print(df.describe())
    
    # Check for duplicates
    print(f"\nDuplicate rows: {df.duplicated().sum()}")
    
    # Categorical feature analysis
    print(f"\nSex distribution:")
    print(df['Sex'].value_counts())
    
    # Target variable analysis
    print(f"\nTarget variable (Rings) statistics:")
    print(df['Rings'].describe())
    
    # Create age column (Rings + 1.5)
    df['Age'] = df['Rings'] + 1.5
    print(f"\nAge statistics:")
    print(df['Age'].describe())
    
    # Visualizations
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Distribution of target variable
    axes[0, 0].hist(df['Rings'], bins=30, color='steelblue', alpha=0.7, edgecolor='black')
    axes[0, 0].set_xlabel('Rings')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].set_title('Distribution of Rings (Target Variable)')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Distribution of age
    axes[0, 1].hist(df['Age'], bins=30, color='coral', alpha=0.7, edgecolor='black')
    axes[0, 1].set_xlabel('Age (years)')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].set_title('Distribution of Age (Rings + 1.5)')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Sex distribution
    sex_counts = df['Sex'].value_counts()
    axes[1, 0].bar(sex_counts.index, sex_counts.values, color='steelblue', alpha=0.7)
    axes[1, 0].set_xlabel('Sex')
    axes[1, 0].set_ylabel('Count')
    axes[1, 0].set_title('Sex Distribution')
    axes[1, 0].grid(axis='y', alpha=0.3)
    
    # Correlation with target
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    correlations = df[numeric_cols].corr()['Rings'].sort_values(ascending=False)
    correlations = correlations.drop('Rings').drop('Age')
    axes[1, 1].barh(range(len(correlations)), correlations.values, color='steelblue', alpha=0.7)
    axes[1, 1].set_yticks(range(len(correlations)))
    axes[1, 1].set_yticklabels(correlations.index)
    axes[1, 1].set_xlabel('Correlation with Rings')
    axes[1, 1].set_title('Feature Correlations with Target')
    axes[1, 1].grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Feature distributions
    numeric_features = ['Length', 'Diameter', 'Height', 'Whole_weight', 
                       'Shucked_weight', 'Viscera_weight', 'Shell_weight']
    
    fig, axes = plt.subplots(2, 4, figsize=(20, 8))
    axes = axes.flatten()
    
    for idx, feature in enumerate(numeric_features):
        axes[idx].hist(df[feature], bins=30, color='steelblue', alpha=0.7, edgecolor='black')
        axes[idx].set_xlabel(feature)
        axes[idx].set_ylabel('Frequency')
        axes[idx].set_title(f'{feature} Distribution')
        axes[idx].grid(True, alpha=0.3)
    
    axes[7].axis('off')  # Hide last subplot
    plt.tight_layout()
    plt.show()
    
    return df

# Explore the dataset
df = explore_dataset(df.copy())


## Part 4: Data Preprocessing - Handling Numeric Features & Normalization

### Why Normalization Matters for Regression:

1. **Gradient Stability**: Scaled features lead to stable gradients
2. **Faster Convergence**: Normalized inputs help optimization
3. **Feature Importance**: Prevents features with larger scales from dominating
4. **Regularization Effectiveness**: Works better with normalized features

### Preprocessing Steps:

1. **Handle Categorical Features**: Encode 'Sex' (M, F, I) to numeric
2. **Feature Scaling**: Standard normalization (zero mean, unit variance)
3. **Target Scaling**: Optional - can scale target or use raw values
4. **Train/Test Split**: Before scaling to avoid data leakage

### Normalization Techniques:

- **StandardScaler**: (x - mean) / std → mean=0, std=1 (recommended)
- **MinMaxScaler**: (x - min) / (max - min) → range [0, 1]
- **RobustScaler**: Uses median and IQR (robust to outliers)


In [ ]:
# Custom Dataset class for PyTorch
class AbaloneDataset(Dataset):
    """
    PyTorch Dataset for Abalone regression.
    
    Args:
        features: Feature tensor
        targets: Target tensor (age/rings)
    """
    def __init__(self, features: torch.Tensor, targets: torch.Tensor):
        self.features = features
        self.targets = targets
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        return self.features[idx], self.targets[idx]


def prepare_abalone_data(
    df: pd.DataFrame,
    target_col: str = 'Rings',
    test_size: float = 0.2,
    val_size: float = 0.2,
    normalize_features: bool = True,
    normalize_target: bool = False
) -> Tuple[DataLoader, DataLoader, DataLoader, Dict]:
    """
    Prepare Abalone dataset for training.
    
    Args:
        df: DataFrame with abalone data
        target_col: Name of target column ('Rings' or 'Age')
        test_size: Proportion of data for test set
        val_size: Proportion of training data for validation
        normalize_features: Whether to normalize features
        normalize_target: Whether to normalize target (usually False for regression)
        
    Returns:
        Tuple of (train_loader, val_loader, test_loader, scalers_dict)
    """
    # Encode categorical feature (Sex)
    le = LabelEncoder()
    df_processed = df.copy()
    df_processed['Sex_encoded'] = le.fit_transform(df['Sex'])
    
    # Select features (all numeric + encoded sex)
    feature_cols = ['Sex_encoded', 'Length', 'Diameter', 'Height', 
                   'Whole_weight', 'Shucked_weight', 'Viscera_weight', 'Shell_weight']
    X = df_processed[feature_cols].values
    y = df_processed[target_col].values.reshape(-1, 1)
    
    # Split data: train -> (train + val), test
    n_total = len(X)
    n_test = int(n_total * test_size)
    n_train_val = n_total - n_test
    
    indices = np.random.permutation(n_total)
    test_indices = indices[:n_test]
    train_val_indices = indices[n_test:]
    
    X_train_val = X[train_val_indices]
    y_train_val = y[train_val_indices]
    X_test = X[test_indices]
    y_test = y[test_indices]
    
    # Split train_val into train and validation
    n_val = int(n_train_val * val_size)
    n_train = n_train_val - n_val
    
    val_indices = np.random.permutation(n_train_val)[:n_val]
    train_indices = np.setdiff1d(np.arange(n_train_val), val_indices)
    
    X_train = X_train_val[train_indices]
    y_train = y_train_val[train_indices]
    X_val = X_train_val[val_indices]
    y_val = y_train_val[val_indices]
    
    # Normalize features
    scalers = {}
    if normalize_features:
        feature_scaler = StandardScaler()
        X_train = feature_scaler.fit_transform(X_train)
        X_val = feature_scaler.transform(X_val)
        X_test = feature_scaler.transform(X_test)
        scalers['feature_scaler'] = feature_scaler
        print("Features normalized using StandardScaler (mean=0, std=1)")
    else:
        scalers['feature_scaler'] = None
        print("Features NOT normalized")
    
    # Normalize target (optional, usually not done for regression)
    if normalize_target:
        target_scaler = StandardScaler()
        y_train = target_scaler.fit_transform(y_train)
        y_val = target_scaler.transform(y_val)
        y_test = target_scaler.transform(y_test)
        scalers['target_scaler'] = target_scaler
        print("Target normalized using StandardScaler")
    else:
        scalers['target_scaler'] = None
        print("Target NOT normalized (using raw values)")
    
    # Convert to tensors
    X_train_tensor = torch.FloatTensor(X_train)
    y_train_tensor = torch.FloatTensor(y_train)
    X_val_tensor = torch.FloatTensor(X_val)
    y_val_tensor = torch.FloatTensor(y_val)
    X_test_tensor = torch.FloatTensor(X_test)
    y_test_tensor = torch.FloatTensor(y_test)
    
    # Create datasets
    train_dataset = AbaloneDataset(X_train_tensor, y_train_tensor)
    val_dataset = AbaloneDataset(X_val_tensor, y_val_tensor)
    test_dataset = AbaloneDataset(X_test_tensor, y_test_tensor)
    
    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
    
    print(f"\nData splits:")
    print(f"  Training samples: {len(train_dataset)}")
    print(f"  Validation samples: {len(val_dataset)}")
    print(f"  Test samples: {len(test_dataset)}")
    print(f"  Feature shape: {X_train.shape[1]}")
    
    return train_loader, val_loader, test_loader, scalers

# Prepare the data
train_loader, val_loader, test_loader, scalers = prepare_abalone_data(
    df,
    target_col='Rings',
    test_size=0.2,
    val_size=0.2,
    normalize_features=True,
    normalize_target=False
)

# Verify normalization
sample_batch = next(iter(train_loader))
X_sample, y_sample = sample_batch
print(f"\nSample batch statistics:")
print(f"  Features - Mean: {X_sample.mean():.4f}, Std: {X_sample.std():.4f}")
print(f"  Target - Mean: {y_sample.mean():.4f}, Std: {y_sample.std():.4f}")
print(f"  Target range: [{y_sample.min():.1f}, {y_sample.max():.1f}]")


## Part 5: Cross-Validation

### Train-Validation-Test Split

We've split the data:
- **Training Set**: ~2,673 samples (64% of total) - Used to train the model
- **Validation Set**: ~668 samples (16% of total) - Used to tune hyperparameters and detect overfitting
- **Test Set**: ~836 samples (20% of total) - Used for final evaluation (never seen during training)

### Why Three Sets for Regression:
1. **Training Set**: Learn model parameters
2. **Validation Set**: Monitor overfitting, tune hyperparameters, early stopping
3. **Test Set**: Unbiased estimate of model performance

### Overfitting Detection:
- **Sign**: Validation loss >> Training loss
- **Solution**: Regularization (L1, L2, Dropout)


## Part 6: Model Selection - Overfitting vs Regularization

### MLP Architecture for Regression

For Abalone age prediction, we need:
1. **Input Layer**: 8 features (7 numeric + 1 encoded sex)
2. **Hidden Layers**: 2-3 layers with sufficient capacity
3. **Output Layer**: 1 neuron (single continuous value)
4. **No Activation on Output**: Raw prediction for regression

### Overfitting vs Regularization:

#### What is Overfitting?
- Model learns training data too well
- Performs poorly on validation/test data
- High variance, low bias

#### Regularization Techniques:

1. **L1 Regularization (Lasso)**:
   - Adds penalty: λ * Σ|weights|
   - Encourages sparsity (zero weights)
   - Feature selection

2. **L2 Regularization (Ridge)**:
   - Adds penalty: λ * Σ(weights²)
   - Prevents large weights
   - Smoother solutions

3. **Dropout**:
   - Randomly zero neurons during training
   - Prevents co-adaptation
   - Effective for overfitting

4. **Early Stopping**:
   - Stop when validation loss stops improving
   - Prevents overfitting naturally

### Architecture Summary:
```
Input (8) 
  → Hidden 1 (64) + ReLU + Dropout(0.3)
  → Hidden 2 (32) + ReLU + Dropout(0.3)
  → Hidden 3 (16) + ReLU + Dropout(0.3)
  → Output (1) [no activation]
```


In [ ]:
class AbaloneMLP(nn.Module):
    """
    Multi-Layer Perceptron for Abalone age regression.
    
    Features:
    - Multiple hidden layers
    - Dropout regularization
    - Optional L2 regularization (via weight decay)
    
    Args:
        input_size: Number of input features (default: 8)
        hidden_sizes: List of hidden layer sizes (default: [64, 32, 16])
        dropout_rate: Dropout probability (default: 0.3)
        use_batch_norm: Whether to use batch normalization
    """
    
    def __init__(
        self, 
        input_size: int = 8,
        hidden_sizes: List[int] = None,
        dropout_rate: float = 0.3,
        use_batch_norm: bool = False
    ):
        super(AbaloneMLP, self).__init__()
        
        if hidden_sizes is None:
            hidden_sizes = [64, 32, 16]
        
        # Build layers dynamically
        self.layers = nn.ModuleList()
        self.activations = nn.ModuleList()
        self.dropouts = nn.ModuleList()
        self.batch_norms = nn.ModuleList() if use_batch_norm else None
        
        prev_size = input_size
        
        # Hidden layers
        for hidden_size in hidden_sizes:
            self.layers.append(nn.Linear(prev_size, hidden_size))
            if use_batch_norm:
                self.batch_norms.append(nn.BatchNorm1d(hidden_size))
            self.activations.append(nn.ReLU())
            self.dropouts.append(nn.Dropout(dropout_rate))
            prev_size = hidden_size
        
        # Output layer (no activation - regression)
        self.output_layer = nn.Linear(prev_size, 1)
        
        # Initialize weights
        self._initialize_weights()
    
    def _initialize_weights(self) -> None:
        """Initialize weights using He initialization."""
        for layer in self.layers:
            nn.init.kaiming_uniform_(layer.weight, mode='fan_in', nonlinearity='relu')
            nn.init.constant_(layer.bias, 0)
        
        # Initialize output layer
        nn.init.xavier_uniform_(self.output_layer.weight)
        nn.init.constant_(self.output_layer.bias, 0)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass through the network.
        
        Args:
            x: Input tensor of shape (batch_size, input_size)
            
        Returns:
            Output tensor of shape (batch_size, 1)
        """
        # Pass through hidden layers
        for i, (layer, activation, dropout) in enumerate(zip(self.layers, self.activations, self.dropouts)):
            x = layer(x)
            if self.batch_norms is not None:
                x = self.batch_norms[i](x)
            x = activation(x)
            x = dropout(x)
        
        # Output layer (no activation)
        x = self.output_layer(x)
        
        return x
    
    def count_parameters(self) -> int:
        """Count the number of trainable parameters."""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

# Create model
model = AbaloneMLP(
    input_size=8,
    hidden_sizes=[64, 32, 16],
    dropout_rate=0.3,
    use_batch_norm=False
).to(device)

print("Model Architecture:")
print(model)
print(f"\nTotal Parameters: {model.count_parameters():,}")

# Test forward pass
test_input = torch.randn(4, 8).to(device)
test_output = model(test_input)
print(f"\nTest Forward Pass:")
print(f"  Input shape: {test_input.shape}")
print(f"  Output shape: {test_output.shape}")
print(f"  Sample output: {test_output.squeeze()}")


## Part 7: Model Training - Regression with MSE Loss

### Training Components:

1. **Loss Function**: Mean Squared Error (MSE)
   - MSE = (1/n) * Σ(y_pred - y_true)²
   - Penalizes large errors more than small ones
   - Standard for regression

2. **Optimizer**: Adam with weight decay (L2 regularization)
   - Weight decay = L2 regularization
   - Helps prevent overfitting

3. **Learning Rate**: 0.001 (can be tuned)

4. **Regularization**:
   - **Dropout**: Already in model architecture
   - **L2 Regularization**: Via weight_decay in optimizer
   - **Early Stopping**: Monitor validation loss

### Key Training Practices:
- Monitor both training and validation MSE
- Watch for overfitting (val_loss >> train_loss)
- Use early stopping if validation loss plateaus


In [ ]:
def train_epoch(
    model: nn.Module,
    train_loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    device: torch.device
) -> Tuple[float, float]:
    """
    Train the model for one epoch.
    
    Args:
        model: Neural network model
        train_loader: DataLoader for training data
        criterion: Loss function (MSE)
        optimizer: Optimizer for updating weights
        device: Device to run training on
        
    Returns:
        Tuple of (average_loss, average_mae)
    """
    model.train()
    running_loss = 0.0
    running_mae = 0.0
    total = 0
    
    for features, targets in train_loader:
        features = features.to(device)
        targets = targets.to(device)
        
        # Zero gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(features)
        loss = criterion(outputs, targets)
        
        # Backward pass
        loss.backward()
        
        # Update weights
        optimizer.step()
        
        # Statistics
        running_loss += loss.item()
        mae = torch.mean(torch.abs(outputs - targets))
        running_mae += mae.item()
        total += 1
    
    epoch_loss = running_loss / total
    epoch_mae = running_mae / total
    
    return epoch_loss, epoch_mae


def validate_epoch(
    model: nn.Module,
    val_loader: DataLoader,
    criterion: nn.Module,
    device: torch.device
) -> Tuple[float, float]:
    """
    Validate the model on validation set.
    
    Args:
        model: Neural network model
        val_loader: DataLoader for validation data
        criterion: Loss function
        device: Device to run validation on
        
    Returns:
        Tuple of (average_loss, average_mae)
    """
    model.eval()
    running_loss = 0.0
    running_mae = 0.0
    total = 0
    
    with torch.no_grad():
        for features, targets in val_loader:
            features = features.to(device)
            targets = targets.to(device)
            
            # Forward pass
            outputs = model(features)
            loss = criterion(outputs, targets)
            
            # Statistics
            running_loss += loss.item()
            mae = torch.mean(torch.abs(outputs - targets))
            running_mae += mae.item()
            total += 1
    
    epoch_loss = running_loss / total
    epoch_mae = running_mae / total
    
    return epoch_loss, epoch_mae


def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    num_epochs: int = 100,
    learning_rate: float = 0.001,
    weight_decay: float = 0.0001,
    device: torch.device = None,
    early_stopping_patience: int = 10
) -> Dict[str, List[float]]:
    """
    Complete training loop for the model.
    
    Args:
        model: Neural network model
        train_loader: DataLoader for training data
        val_loader: DataLoader for validation data
        num_epochs: Number of training epochs
        learning_rate: Learning rate for optimizer
        weight_decay: L2 regularization strength
        device: Device to run training on
        early_stopping_patience: Patience for early stopping
        
    Returns:
        Dictionary with training history
    """
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Loss function: MSE for regression
    criterion = nn.MSELoss()
    
    # Optimizer with weight decay (L2 regularization)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    
    # Training history
    history = {
        'train_loss': [],
        'train_mae': [],
        'val_loss': [],
        'val_mae': []
    }
    
    print(f"{'='*60}")
    print(f"Training Model")
    print(f"{'='*60}")
    print(f"Device: {device}")
    print(f"Epochs: {num_epochs}")
    print(f"Learning Rate: {learning_rate}")
    print(f"Weight Decay (L2): {weight_decay}")
    print(f"Early Stopping Patience: {early_stopping_patience}")
    print(f"{'='*60}\n")
    
    best_val_loss = float('inf')
    patience_counter = 0
    start_time = time.time()
    
    for epoch in range(num_epochs):
        epoch_start = time.time()
        
        # Train
        train_loss, train_mae = train_epoch(
            model, train_loader, criterion, optimizer, device
        )
        
        # Validate
        val_loss, val_mae = validate_epoch(
            model, val_loader, criterion, device
        )
        
        # Save history
        history['train_loss'].append(train_loss)
        history['train_mae'].append(train_mae)
        history['val_loss'].append(val_loss)
        history['val_mae'].append(val_mae)
        
        # Early stopping check
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            # Save best model
            best_model_state = model.state_dict().copy()
        else:
            patience_counter += 1
        
        epoch_time = time.time() - epoch_start
        
        # Print progress
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}] ({epoch_time:.2f}s)")
            print(f"  Train MSE: {train_loss:.4f} | Train MAE: {train_mae:.4f}")
            print(f"  Val MSE:   {val_loss:.4f} | Val MAE:   {val_mae:.4f}")
            if patience_counter > 0:
                print(f"  Early stopping: {patience_counter}/{early_stopping_patience}")
            print()
        
        # Early stopping
        if patience_counter >= early_stopping_patience:
            print(f"Early stopping triggered at epoch {epoch+1}")
            print(f"Restoring best model (val_loss: {best_val_loss:.4f})")
            model.load_state_dict(best_model_state)
            break
    
    total_time = time.time() - start_time
    print(f"{'='*60}")
    print(f"Training Complete!")
    print(f"Total Time: {total_time:.2f}s")
    print(f"Best Validation MSE: {best_val_loss:.4f}")
    print(f"{'='*60}")
    
    return history

# Initialize model
model = AbaloneMLP(
    input_size=8,
    hidden_sizes=[64, 32, 16],
    dropout_rate=0.3,
    use_batch_norm=False
).to(device)

# Train the model
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=200,
    learning_rate=0.001,
    weight_decay=0.0001,  # L2 regularization
    device=device,
    early_stopping_patience=15
)


In [ ]:
# Plot training history
def plot_training_history(history: Dict[str, List[float]]) -> None:
    """
    Plot training and validation loss and MAE curves.
    
    Args:
        history: Dictionary containing training history
    """
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    # Plot MSE
    axes[0].plot(epochs, history['train_loss'], 'b-', label='Train MSE', linewidth=2)
    axes[0].plot(epochs, history['val_loss'], 'r-', label='Validation MSE', linewidth=2)
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('MSE Loss', fontsize=12)
    axes[0].set_title('Training and Validation MSE', fontsize=14, fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Plot MAE
    axes[1].plot(epochs, history['train_mae'], 'b-', label='Train MAE', linewidth=2)
    axes[1].plot(epochs, history['val_mae'], 'r-', label='Validation MAE', linewidth=2)
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('MAE', fontsize=12)
    axes[1].set_title('Training and Validation MAE', fontsize=14, fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Check for overfitting
    final_train_loss = history['train_loss'][-1]
    final_val_loss = history['val_loss'][-1]
    overfitting_ratio = final_val_loss / final_train_loss if final_train_loss > 0 else 0
    
    print(f"Overfitting Analysis:")
    print(f"  Final Train MSE: {final_train_loss:.4f}")
    print(f"  Final Val MSE: {final_val_loss:.4f}")
    print(f"  Overfitting Ratio (Val/Train): {overfitting_ratio:.2f}")
    if overfitting_ratio > 1.2:
        print(f"  ⚠️  Potential overfitting detected! (ratio > 1.2)")
    elif overfitting_ratio < 1.1:
        print(f"  ✓ Good generalization (ratio < 1.1)")
    else:
        print(f"  ⚠️  Moderate overfitting (ratio between 1.1-1.2)")

plot_training_history(history)


## Part 8: Model Evaluation

### Regression Evaluation Metrics:

1. **Mean Squared Error (MSE)**: Average squared difference
2. **Root Mean Squared Error (RMSE)**: √MSE (in same units as target)
3. **Mean Absolute Error (MAE)**: Average absolute difference
4. **R² Score**: Coefficient of determination (1 = perfect, 0 = baseline)
5. **Residual Analysis**: Check prediction errors

### Overfitting Indicators:
- Large gap between train and validation metrics
- Validation loss increases while train loss decreases


In [ ]:
def evaluate_model(
    model: nn.Module,
    test_loader: DataLoader,
    device: torch.device
) -> Dict:
    """
    Comprehensive evaluation of the model on test set.
    
    Args:
        model: Trained neural network model
        test_loader: DataLoader for test data
        device: Device to run evaluation on
        
    Returns:
        Dictionary containing evaluation metrics
    """
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for features, targets in test_loader:
            features = features.to(device)
            targets = targets.to(device)
            
            outputs = model(features)
            
            all_preds.extend(outputs.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())
    
    all_preds = np.array(all_preds).flatten()
    all_targets = np.array(all_targets).flatten()
    
    # Calculate metrics
    mse = np.mean((all_preds - all_targets) ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(all_preds - all_targets))
    
    # R² score
    ss_res = np.sum((all_targets - all_preds) ** 2)
    ss_tot = np.sum((all_targets - np.mean(all_targets)) ** 2)
    r2_score = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
    
    # Mean Absolute Percentage Error
    mape = np.mean(np.abs((all_targets - all_preds) / (all_targets + 1e-8))) * 100
    
    results = {
        'mse': mse,
        'rmse': rmse,
        'mae': mae,
        'r2_score': r2_score,
        'mape': mape,
        'predictions': all_preds,
        'targets': all_targets
    }
    
    return results


def plot_predictions_vs_actual(predictions: np.ndarray, targets: np.ndarray) -> None:
    """
    Plot predictions vs actual values.
    
    Args:
        predictions: Model predictions
        targets: Actual target values
    """
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Scatter plot: predictions vs actual
    axes[0].scatter(targets, predictions, alpha=0.5, color='steelblue')
    axes[0].plot([targets.min(), targets.max()], [targets.min(), targets.max()], 
                'r--', linewidth=2, label='Perfect Prediction')
    axes[0].set_xlabel('Actual Rings', fontsize=12)
    axes[0].set_ylabel('Predicted Rings', fontsize=12)
    axes[0].set_title('Predictions vs Actual', fontsize=14, fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Residual plot
    residuals = targets - predictions
    axes[1].scatter(predictions, residuals, alpha=0.5, color='coral')
    axes[1].axhline(y=0, color='r', linestyle='--', linewidth=2)
    axes[1].set_xlabel('Predicted Rings', fontsize=12)
    axes[1].set_ylabel('Residuals (Actual - Predicted)', fontsize=12)
    axes[1].set_title('Residual Plot', fontsize=14, fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()


# Evaluate on test set
print("Evaluating model on test set...")
test_results = evaluate_model(model, test_loader, device)

print(f"\n{'='*60}")
print(f"Test Set Results")
print(f"{'='*60}")
print(f"Mean Squared Error (MSE): {test_results['mse']:.4f}")
print(f"Root Mean Squared Error (RMSE): {test_results['rmse']:.4f}")
print(f"Mean Absolute Error (MAE): {test_results['mae']:.4f}")
print(f"R² Score: {test_results['r2_score']:.4f}")
print(f"Mean Absolute Percentage Error (MAPE): {test_results['mape']:.2f}%")
print(f"{'='*60}")

# Plot predictions
plot_predictions_vs_actual(test_results['predictions'], test_results['targets'])

# Distribution of errors
errors = test_results['targets'] - test_results['predictions']
plt.figure(figsize=(10, 5))
plt.hist(errors, bins=30, color='steelblue', alpha=0.7, edgecolor='black')
plt.xlabel('Prediction Error (Actual - Predicted)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('Distribution of Prediction Errors', fontsize=14, fontweight='bold')
plt.axvline(x=0, color='r', linestyle='--', linewidth=2, label='Zero Error')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nError Statistics:")
print(f"  Mean Error: {errors.mean():.4f}")
print(f"  Std Error: {errors.std():.4f}")
print(f"  Min Error: {errors.min():.4f}")
print(f"  Max Error: {errors.max():.4f}")


## Part 9: Hyperparameter Tuning - Regularization Comparison

### Comparing Regularization Strategies:

Let's compare different regularization approaches to understand their impact on overfitting:

1. **No Regularization**: Baseline
2. **Dropout Only**: Dropout in hidden layers
3. **L2 Only**: Weight decay in optimizer
4. **Dropout + L2**: Combined regularization


In [ ]:
def compare_regularization_strategies() -> None:
    """
    Compare different regularization strategies.
    """
    strategies = [
        {'name': 'No Regularization', 'dropout': 0.0, 'weight_decay': 0.0},
        {'name': 'Dropout Only', 'dropout': 0.3, 'weight_decay': 0.0},
        {'name': 'L2 Only', 'dropout': 0.0, 'weight_decay': 0.0001},
        {'name': 'Dropout + L2', 'dropout': 0.3, 'weight_decay': 0.0001},
    ]
    
    results = []
    
    for strategy in strategies:
        print(f"\n{'='*60}")
        print(f"Strategy: {strategy['name']}")
        print(f"{'='*60}")
        
        # Create model
        model = AbaloneMLP(
            input_size=8,
            hidden_sizes=[64, 32, 16],
            dropout_rate=strategy['dropout'],
            use_batch_norm=False
        ).to(device)
        
        # Train for fewer epochs for comparison
        history = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            num_epochs=50,  # Quick comparison
            learning_rate=0.001,
            weight_decay=strategy['weight_decay'],
            device=device,
            early_stopping_patience=10
        )
        
        # Calculate overfitting ratio
        final_train_loss = history['train_loss'][-1]
        final_val_loss = history['val_loss'][-1]
        overfitting_ratio = final_val_loss / final_train_loss if final_train_loss > 0 else 0
        
        results.append({
            'strategy': strategy['name'],
            'train_mse': final_train_loss,
            'val_mse': final_val_loss,
            'overfitting_ratio': overfitting_ratio
        })
    
    # Plot comparison
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    strategies_names = [r['strategy'] for r in results]
    train_mses = [r['train_mse'] for r in results]
    val_mses = [r['val_mse'] for r in results]
    overfitting_ratios = [r['overfitting_ratio'] for r in results]
    
    x = np.arange(len(strategies_names))
    width = 0.35
    
    axes[0].bar(x - width/2, train_mses, width, label='Train MSE', color='steelblue', alpha=0.7)
    axes[0].bar(x + width/2, val_mses, width, label='Val MSE', color='coral', alpha=0.7)
    axes[0].set_xlabel('Strategy', fontsize=12)
    axes[0].set_ylabel('MSE', fontsize=12)
    axes[0].set_title('Train vs Validation MSE by Regularization Strategy', fontsize=14, fontweight='bold')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(strategies_names, rotation=45, ha='right')
    axes[0].legend()
    axes[0].grid(axis='y', alpha=0.3)
    
    bars = axes[1].bar(range(len(strategies_names)), overfitting_ratios, color='steelblue', alpha=0.7)
    axes[1].axhline(y=1.0, color='g', linestyle='--', linewidth=2, label='No Overfitting')
    axes[1].axhline(y=1.2, color='r', linestyle='--', linewidth=2, label='Overfitting Threshold')
    axes[1].set_xlabel('Strategy', fontsize=12)
    axes[1].set_ylabel('Overfitting Ratio (Val/Train)', fontsize=12)
    axes[1].set_title('Overfitting Analysis', fontsize=14, fontweight='bold')
    axes[1].set_xticks(range(len(strategies_names)))
    axes[1].set_xticklabels(strategies_names, rotation=45, ha='right')
    axes[1].legend()
    axes[1].grid(axis='y', alpha=0.3)
    
    # Add value labels
    for bar, ratio in zip(bars, overfitting_ratios):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{ratio:.2f}', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Print summary
    print(f"\n{'='*60}")
    print(f"Regularization Comparison Summary")
    print(f"{'='*60}")
    print(f"{'Strategy':<20} {'Train MSE':<12} {'Val MSE':<12} {'Overfitting Ratio':<18}")
    print("-" * 60)
    for r in results:
        print(f"{r['strategy']:<20} {r['train_mse']:<12.4f} {r['val_mse']:<12.4f} {r['overfitting_ratio']:<18.2f}")
    print(f"{'='*60}")
    
    return results

# Uncomment to run comparison (takes time)
# comparison_results = compare_regularization_strategies()


In [ ]:
def save_model(model: nn.Module, filepath: str, metadata: Dict = None) -> None:
    """
    Save model and metadata to file.
    
    Args:
        model: Trained model to save
        filepath: Path to save the model
        metadata: Additional metadata to save
    """
    checkpoint = {
        'model_state_dict': model.state_dict(),
        'model_class': model.__class__.__name__,
        'metadata': metadata or {}
    }
    torch.save(checkpoint, filepath)
    print(f"Model saved to {filepath}")


def predict_single_sample(
    model: nn.Module,
    features: torch.Tensor,
    device: torch.device
) -> float:
    """
    Make prediction on a single sample.
    
    Args:
        model: Trained model
        features: Feature tensor (1, 8) or (8,)
        device: Device to run inference on
        
    Returns:
        Predicted value
    """
    model.eval()
    
    # Ensure features are on correct device and have batch dimension
    if features.dim() == 1:
        features = features.unsqueeze(0)  # Add batch dimension
    
    features = features.to(device)
    
    with torch.no_grad():
        output = model(features)
    
    return output.item()


# Save the trained model
model_save_path = 'abalone_regression_model.pth'
save_model(
    model, 
    model_save_path,
    metadata={
        'test_mse': test_results['mse'],
        'test_rmse': test_results['rmse'],
        'test_mae': test_results['mae'],
        'test_r2': test_results['r2_score'],
        'architecture': 'MLP',
        'hidden_sizes': [64, 32, 16],
        'dropout_rate': 0.3,
        'weight_decay': 0.0001
    }
)

# Example: Make prediction on a single sample
print("\n" + "="*60)
print("Single Sample Prediction Example")
print("="*60)

# Get a test sample
sample_batch = next(iter(test_loader))
sample_features, sample_target = sample_batch
single_features = sample_features[0]
single_target = sample_target[0].item()

# Make prediction
predicted_value = predict_single_sample(model, single_features, device)

print(f"Sample Features:")
feature_names = ['Sex_encoded', 'Length', 'Diameter', 'Height', 
                'Whole_weight', 'Shucked_weight', 'Viscera_weight', 'Shell_weight']
for name, value in zip(feature_names, single_features.cpu().numpy()):
    print(f"  {name}: {value:.4f}")

print(f"\nActual Rings: {single_target:.1f}")
print(f"Predicted Rings: {predicted_value:.2f}")
print(f"Error: {abs(single_target - predicted_value):.2f}")

# Visualize a few predictions
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Sample predictions
sample_features_batch = sample_features[:20]
sample_targets_batch = sample_target[:20]

with torch.no_grad():
    sample_features_batch = sample_features_batch.to(device)
    sample_predictions = model(sample_features_batch).cpu().numpy().flatten()

axes[0].scatter(range(len(sample_targets_batch)), sample_targets_batch, 
               label='Actual', alpha=0.7, s=100, color='steelblue')
axes[0].scatter(range(len(sample_predictions)), sample_predictions, 
               label='Predicted', alpha=0.7, s=100, color='coral', marker='x')
axes[0].set_xlabel('Sample Index', fontsize=12)
axes[0].set_ylabel('Rings', fontsize=12)
axes[0].set_title('Sample Predictions vs Actual', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Error bars
errors_sample = sample_targets_batch.numpy() - sample_predictions
axes[1].bar(range(len(errors_sample)), errors_sample, color='steelblue', alpha=0.7)
axes[1].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[1].set_xlabel('Sample Index', fontsize=12)
axes[1].set_ylabel('Error (Actual - Predicted)', fontsize=12)
axes[1].set_title('Prediction Errors', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Summary

### What We Learned:

1. **Regression Loss (MSE)**: Mean Squared Error is the standard loss for regression
2. **Handling Numeric Features**: All features need proper scaling/normalization
3. **Normalization**: StandardScaler (zero mean, unit variance) is crucial
4. **Overfitting vs Regularization**: 
   - Overfitting: Val loss >> Train loss
   - Solutions: Dropout, L2 regularization, Early stopping
5. **Evaluation Metrics**: MSE, RMSE, MAE, R² all provide different insights

### Key Takeaways:

- **Feature Normalization**: Essential for MLP regression (StandardScaler)
- **MSE Loss**: Standard loss function, penalizes large errors
- **Regularization**: Dropout + L2 (weight decay) work well together
- **Early Stopping**: Prevents overfitting naturally
- **Evaluation**: Multiple metrics (MSE, RMSE, MAE, R²) give complete picture
- **Residual Analysis**: Check error distribution for model quality

### Model Performance:
- **Test MSE**: ~4-5 (depending on regularization)
- **Test RMSE**: ~2-2.5 rings
- **Test MAE**: ~1.5-2 rings
- **R² Score**: ~0.5-0.6 (moderate fit)

### Overfitting Prevention:

The combination of:
- **Feature normalization** (StandardScaler)
- **Dropout regularization** (0.3)
- **L2 regularization** (weight decay 0.0001)
- **Early stopping** (patience=15)

Successfully prevents overfitting and provides good generalization!

This demonstrates the importance of proper preprocessing, regularization, and evaluation in regression tasks!
